In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import os
import warnings
from tqdm import tqdm
import pickle

In [2]:
mapping = {'albite':1,
           'an':8,
           'aug':6,
           'bt':3,
           'cal':10,
           'en':7,
           'fo':5,
           'foalb':14,
           'foaug':13,
           'gyp':11,
           'hb':2,
           'mc':9,
           'ms':4,
           'qtz':0,
           'qtzalb':12,
           'Unknown':15}

rev_mapping = {v:k for k,v in mapping.items()}

In [3]:
#Saving the names of all the csv files in the current directory
all_files = os.listdir()
test_csv_files = [x for x in all_files if x.endswith(".csv") and "test" in x]
train_csv_files = [x for x in all_files if x.endswith(".csv") and "train" in x]

print(f"The number of test files are {len(test_csv_files)} and train files are {len(train_csv_files)}")

The number of test files are 7 and train files are 34


In [4]:
#Key is the name of the csv file and the value is the values in the file
df_mapping = {}
for csv in test_csv_files+train_csv_files:
    warnings.filterwarnings("ignore")
    df = pd.read_csv(os.path.join(csv), header=None) #Reading the value of the csv file

    x_axis = (np.array(df))[0,1:-1].astype("float64") #The wavenumbers 
    X = (np.array(df))[1:,1:-1].astype("float64") #The spectra
    y = (np.array(df))[1:,-1].astype("int64") #The class labels

    df_mapping[csv.split(".")[0]] = [x_axis,X,y]
    print(f"{csv} has {X.shape[1]} frequency components and {X.shape[0]} elements")

granite50dust_test_015s_10000.csv has 2064 frequency components and 10000 elements
granite0dust_test_030s_1373_final.csv has 2064 frequency components and 1373 elements
granite0dust_test_015s_5571_final.csv has 2064 frequency components and 5571 elements
gabbro50dust_test_015s_9740.csv has 1371 frequency components and 9740 elements
granite0dust_test_015s_4084_final.csv has 2184 frequency components and 4084 elements
gabbro0dust_test_015s_8906_final.csv has 1371 frequency components and 8906 elements
gabbro0dust_test_015s_46_final.csv has 1371 frequency components and 46 elements
bt_train_015s_1100.csv has 2064 frequency components and 1100 elements
albite_train_015s_625.csv has 2184 frequency components and 625 elements
qtz_train_003s_1148.csv has 3670 frequency components and 1148 elements
fo_train_030s_5040.csv has 2064 frequency components and 5040 elements
an_train_015s_5600.csv has 2290 frequency components and 5600 elements
aug_train_20s_5103.csv has 1195 frequency components an

In [5]:
max_freq = 2000.0
min_freq = 0.0
for key,(x,_,_) in df_mapping.items():
    max_freq = min(max_freq,x[-1])
    min_freq = max(min_freq,x[0])
    print(f"{key} has {x[0]}Hz as the minimum and {x[-1]}Hz as the maximum frequency components")

print(f"\n The highest minimum frequency is {min_freq}Hz and lowest maximum frequency is {max_freq}Hz")

granite50dust_test_015s_10000 has 95.4877Hz as the minimum and 1100.97Hz as the maximum frequency components
granite0dust_test_030s_1373_final has 95.4877Hz as the minimum and 1100.97Hz as the maximum frequency components
granite0dust_test_015s_5571_final has 95.4877Hz as the minimum and 1100.97Hz as the maximum frequency components
gabbro50dust_test_015s_9740 has 96.5973Hz as the minimum and 1204.94Hz as the maximum frequency components
granite0dust_test_015s_4084_final has 100.473Hz as the minimum and 1800.29Hz as the maximum frequency components
gabbro0dust_test_015s_8906_final has 96.5973Hz as the minimum and 1204.94Hz as the maximum frequency components
gabbro0dust_test_015s_46_final has 96.5973Hz as the minimum and 1204.94Hz as the maximum frequency components
bt_train_015s_1100 has 95.4877Hz as the minimum and 1100.97Hz as the maximum frequency components
albite_train_015s_625 has 100.473Hz as the minimum and 1800.99Hz as the maximum frequency components
qtz_train_003s_1148 has 

In [6]:
#interpolating to new spectral region
new_df_mapping = {}
new_x_axis = np.linspace(min_freq,max_freq,1024) #of shape 1024
for key,(x_axis,x,y) in df_mapping.items():
    new_vals = []
    for i in range(x.shape[0]):
        z = np.interp(new_x_axis,x_axis,x[i])[np.newaxis, ...] #of shape (1,1024)
        new_vals.append(z)    
    
    new_vals = np.stack(new_vals) 
    new_df_mapping[key] = [new_vals,y]
    print(f"{key} has {new_vals.shape[-1]} frequency components and {new_vals.shape[0]} elements")

granite50dust_test_015s_10000 has 1024 frequency components and 10000 elements
granite0dust_test_030s_1373_final has 1024 frequency components and 1373 elements
granite0dust_test_015s_5571_final has 1024 frequency components and 5571 elements
gabbro50dust_test_015s_9740 has 1024 frequency components and 9740 elements
granite0dust_test_015s_4084_final has 1024 frequency components and 4084 elements
gabbro0dust_test_015s_8906_final has 1024 frequency components and 8906 elements
gabbro0dust_test_015s_46_final has 1024 frequency components and 46 elements
bt_train_015s_1100 has 1024 frequency components and 1100 elements
albite_train_015s_625 has 1024 frequency components and 625 elements
qtz_train_003s_1148 has 1024 frequency components and 1148 elements
fo_train_030s_5040 has 1024 frequency components and 5040 elements
an_train_015s_5600 has 1024 frequency components and 5600 elements
aug_train_20s_5103 has 1024 frequency components and 5103 elements
aug_train_15s_1000 has 1024 frequenc

In [7]:
train_spectra = []
train_labels = []

test_spectra = []
test_labels = []

for key,(x,y) in new_df_mapping.items():
    if "train" in key:
        train_spectra.append(x)
        train_labels.append(y)
    
    elif "test" in key:
        test_spectra.append(x)
        test_labels.append(y)
        
train_spectra = np.vstack(train_spectra)
test_spectra = np.vstack(test_spectra)

train_labels = np.concatenate(train_labels)
test_labels = np.concatenate(test_labels)

In [8]:
train_count = {x:0 for x in range(16)}
test_count = {x:0 for x in range(16)}

for i in range(len(train_labels)):
    train_count[train_labels[i]]+=1 

for i in range(len(test_labels)):
    test_count[test_labels[i]]+=1


In [9]:
print("In the training dataset")
for i in range(16):
    print(f"{rev_mapping[i]}: {train_count[i]}")

print("\nIn the test dataset")
for i in range(16):
    print(f"{rev_mapping[i]}: {test_count[i]}")

In the training dataset
qtz: 10032
albite: 6975
hb: 5054
bt: 7925
ms: 5900
fo: 6140
aug: 6103
en: 5600
an: 5600
mc: 5600
cal: 5032
gyp: 5000
qtzalb: 5000
foaug: 5160
foalb: 5220
Unknown: 0

In the test dataset
qtz: 7921
albite: 3447
hb: 901
bt: 208
ms: 504
fo: 2449
aug: 8291
en: 132
an: 3537
mc: 6488
cal: 800
gyp: 16
qtzalb: 132
foaug: 44
foalb: 33
Unknown: 4817


In [10]:
with open("MLROD_train.pkl","wb") as file:
    pickle.dump([train_labels,train_spectra],file) 

with open("MLROD_test.pkl","wb") as file:
    pickle.dump([test_labels,test_spectra],file) 